<h1>Agents and Workflows (14.7%)</h1>

`Lab 3.1: Hand-roll an agentic loop

In [2]:
import sys
from pathlib import Path
import anthropic
import time

sys.path.insert(0, str(Path.cwd().parent))

from common import get_client, MODEL

client = get_client()

tools = [{"name": "get_time", "description": "Current UTC time", "input_schema": {"type": "object", "properties": {}}}]
messages = [{"role": "user", "content": "What time is it?"}]
while True:
    resp = client.messages.create(model="claude-sonnet-4-5", max_tokens=300, tools=tools, messages=messages)
    messages.append({"role": "assistant", "content": resp.content})
    if resp.stop_reason != "tool_use":
        break
    tool_block = next(b for b in resp.content if b.type == "tool_use")
    result = "2026-10-15T12:00:00Z"  # your real tool logic goes here
    messages.append({"role": "user", "content": [{"type": "tool_result", "tool_use_id": tool_block.id, "content": result}]})
print(resp.content[0].text)

The current UTC time is **12:00:00 PM (noon) on October 15, 2026**.


` Lab 3.2: Multi-step workflow
1. Chain two agents: a "planner" call that outputs a numbered plan, and an "executor" loop (Lab 3.1's pattern) that works through each  

2. Log every intermediate step so you can debug the chain later — this doubles as prep for Domain 8.

In [ ]:

import re

from common import get_client, MODEL, banner

client = get_client()

###Mock Order Items 

##Instead of order i am trying to define three products in semiconductor and its status

PRODUCTS = {

    "PRD-001":{"status":"qualified","custoemr":"AAA"},
    "PRD-002":{"status":"pending","customer":"BBB"},
    "PRD-003":{"status":"qualified","customer":"CCC"}

}

TOOLS = [

    {

        "name":"get_product_status",
        "description":"Look up the status of single product by its ID",
        "input_schema":{
            "type": "object",
            "properties": {"product_id":{"type":"string"}},
            "required":["product_id"]
        }

    },


     {

        "name":"search_products",
        "description":"Find all orders matching a given status",
        "input_schema":{
            "type": "object",
            "properties": {"product_id":{"type":"string","enum":["pending","qualified"]}},
            "required":["product_id"]
        }

    }

]

def _get_product_status(product_id:str) -> str:
    product = PRODUCTS.get(product_id)
    return product["status"] if product else f"No such product id: {product_id}"

def _search_products(status:str) ->str:
    matches = [pid for pid,o in PRODUCTS.items() if o['status'] == status]
    return ", " .join(matches) if matches else f"No Products with Status {status}"


TOOL_IMPLS = {"get_product_status": _get_product_status, "search_products": _search_products}

# --- Executor: Lab 3.1's loop, generalized to run on any single-step task ---
def run_executor(task: str, max_iterations: int = 5) -> str:
    messages = [{"role": "user", "content": task}]
    for _ in range(max_iterations):
        resp = client.messages.create(
            model=MODEL, max_tokens=500, tools=TOOLS, messages=messages,
        )
        messages.append({"role": "assistant", "content": resp.content})

        if resp.stop_reason != "tool_use":
            text_block = next(b for b in resp.content if b.type == "text")
            return text_block.text

        # Handle every tool_use block in this turn, not just the first --
        # Claude can request more than one tool call per turn.
        tool_results = []
        for block in resp.content:
            if block.type != "tool_use":
                continue
            impl = TOOL_IMPLS.get(block.name)
            result = impl(**block.input) if impl else f"Unknown tool: {block.name}"
            print(f"Results from the Tool Implementation: {result}")
            tool_results.append(
                {"type": "tool_result", "tool_use_id": block.id, "content": result}
            )
            print(f"Tools_Results : {tool_results}")
        messages.append({"role": "user", "content": tool_results})
 
    return "Executor hit max_iterations without finishing."


# --- Planner: one call, numbered-list output, parsed with regex ---
PLANNER_SYSTEM = (
    "You are a planner. Break the user's goal into a short numbered list of "
    "concrete, independently-executable steps. Output ONLY the numbered "
    "list, one step per line, nothing else."
)
 

def run_planner(goal: str) -> list[str]:
    resp = client.messages.create(
        model=MODEL,
        max_tokens=300,
        system=PLANNER_SYSTEM,
        messages=[{"role": "user", "content": goal}],
    )
    plan_text = resp.content[0].text
    return re.findall(r"^\s*\d+\.\s*(.+)$", plan_text, re.MULTILINE)


# --- Chain: planner's output drives one executor run per step ---
def run_chain(goal: str) -> list[dict]:
    banner(f"PLANNER: {goal}")
    steps = run_planner(goal)
    for i, step in enumerate(steps, 1):
        print(f"  {i}. {step}")
 
    log = []
    for i, step in enumerate(steps, 1):
        banner(f"EXECUTOR step {i}/{len(steps)}")
        print(f"Task: {step}")
        result = run_executor(step)
        print(f"Result: {result}")
        log.append({"step": step, "result": result})
    return log



if __name__ == "__main__":
    run_chain(
        "Check whether product PRD-001 has qualified, and if not, list"
        "other pending product so we know what else needs attention."
    )


In [19]:
_search_products("PPPP")

'No Products with Status PPPP'

In [ ]:
run_executor("Get the porduct id details of status - pending")

In [10]:
matches = [pid for pid,o in PRODUCTS.items() if o['status'] == 'qualified' or 'pending']

In [11]:
matches

['PRD-001', 'PRD-002', 'PRD-003']